#### Relevant Imports

# Notebook 1 — Connecting to MySQL & Multi-Table Queries

## What You Will Learn

This notebook introduces how to connect to a remote relational database from Python and retrieve data spanning multiple tables.

### Topics Covered

| Topic | Description |
|---|---|
| **SQLAlchemy Engine** | Database-agnostic abstraction that manages connections to any SQL backend |
| **Multiple Connections** | Creating and using several concurrent connections from a single engine |
| **SQL Queries** | Executing raw SQL via `text()` and iterating over results |
| **JOIN Operations** | Consolidating data across 3+ tables using foreign-key relationships |

### Skills You Will Build

- Connect to a remote MySQL database (RFAM — RNA family database) using `SQLAlchemy`
- Understand the `Engine → Connection → Query` lifecycle
- Write `JOIN` queries to pull coherent records from multiple tables
- Inspect row objects and fetch result data programmatically

> **Pre-requisite:** Basic SQL knowledge (SELECT, WHERE, JOIN). No prior SQLAlchemy experience needed.

In [1]:
from sqlalchemy import create_engine, text

**Engine**  
Engine is the abstract object from SQLAlchmey which connects to a host to particular Database  
In an an engine further connections can be created to query data  
This generic concepts make it Database agnostic

# goal: work with/query databases in python
# sql

In [2]:
# There is an engine instance created, which can handle multiple connetions
sql_engine = create_engine("mysql+pymysql://rfamro:@mysql-rfam-public.ebi.ac.uk:4497/Rfam")

**Connection and Query**
Connection object creates a connection (its within the object)  
Then it is used to raise query (Execute Query String)

In [3]:
# Create a Connection in engine and raise a query to get data
conn_1 = sql_engine.connect()
result = conn_1.execute(text("SELECT COUNT(*) FROM full_region;"))

# Fetch the result content
for row in result:
    print (row)
    print (type(row))


(10645620,)
<class 'sqlalchemy.engine.row.Row'>


In [4]:
# Get more data, parallelly from another connection in same engine
conn_2 = sql_engine.connect ()
result = conn_2.execute(text("SELECT * FROM full_region LIMIT 10;"))

# Fetch the result content
print (result.keys())
for row in result:

    print (row)
    print (type(row))


RMKeyView(['rfam_acc', 'rfamseq_acc', 'seq_start', 'seq_end', 'bit_score', 'evalue_score', 'cm_start', 'cm_end', 'truncated', 'type', 'is_significant'])
('RF00061', 'AF009606.1', 2, 354, 434.2, '3.8e-132', 1, 352, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'AF029248.1', 20265, 20371, 128.5, '7.6e-24', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'AF201929.1', 20103, 20209, 128.5, '7.6e-24', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'AF207902.1', 20103, 20209, 128.5, '7.6e-24', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'JN874562.1', 20063, 20172, 123.1, '2e-22', 1, 107, '0', 'seed', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'FJ425184.1', 19998, 20104, 122.0, '3.9e-22', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'FJ425187.1', 20000, 20106, 122.0, '3.9e-22', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'DQ339101.1', 20

In [5]:
# For a Given value of rfam_acc, how many Ncbi Id
result = conn_1.execute (text(
                             "SELECT COUNT(DISTINCT fn.ncbi_id) AS ncbi_count\
                            FROM family_ncbi fn\
                            WHERE fn.rfam_acc = 'RF01530';"))

for row in result:
    print (row)


(14,)


**Data from multiple tables**  
Data from multiple table can be collated by the linking fields by JOIN queries

In [6]:
# For a Given value of rfam_acc, how many species are present in another table based on its ncbi_id
result = conn_1.execute (text(
                             "SELECT DISTINCT fn.rfam_acc, fn.ncbi_id, t.species AS species\
                            FROM family_ncbi fn\
                            JOIN taxonomy t ON fn.ncbi_id = t.ncbi_id\
                            WHERE fn.rfam_acc = 'RF01530'"))

for row in result:
    print (row) 


('RF01530', 190650, 'Caulobacter crescentus CB15')
('RF01530', 565050, 'Caulobacter crescentus NA1000')
('RF01530', 1736578, 'Caulobacter sp. Root655')
('RF01530', 2172650, 'Caulobacter radicis')
('RF01530', 1813876, 'Phenylobacterium hankyongense')
('RF01530', 1445034, 'Phenylobacterium kunshanense')
('RF01530', 69395, 'Caulobacter henricii')
('RF01530', 2170551, 'Phenylobacterium soli')
('RF01530', 2015570, 'Alphaproteobacteria bacterium PA2')
('RF01530', 2803784, 'Phenylobacterium glaciei')
('RF01530', 1736442, 'Phenylobacterium sp. Root1277')
('RF01530', 1914756, 'Phenylobacterium deserti')
('RF01530', 450851, 'Phenylobacterium zucineum HLK1')
('RF01530', 69666, 'Caulobacter sp. FWC38')


In [7]:
# Consolidated information from 3 tables
result = conn_1.execute (text(
                                "SELECT fn.rfam_id, fn.ncbi_id, t.species, f.rfam_id AS family_rfam_id, f.auto_wiki, f.description\
                                FROM family_ncbi fn\
                                JOIN taxonomy t ON fn.ncbi_id = t.ncbi_id\
                                JOIN family f ON fn.rfam_acc = f.rfam_acc\
                                WHERE fn.rfam_acc = 'RF01530'"))

rows = result.fetchall ()
print (rows)
  

[('CC3664', 190650, 'Caulobacter crescentus CB15', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 565050, 'Caulobacter crescentus NA1000', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 1736578, 'Caulobacter sp. Root655', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 2172650, 'Caulobacter radicis', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 1813876, 'Phenylobacterium hankyongense', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 1445034, 'Phenylobacterium kunshanense', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 69395, 'Caulobacter henricii', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 2170551, 'Phenylobacterium soli', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 2015570, 'Alphaproteobacteria bacterium PA2', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 2803784, 'Phenylobacterium glaciei', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 1736442, 'Phenylobacterium sp. Root1277', 'CC3664', 2085, '

In [8]:
import pandas as pd

df = pd.DataFrame(rows, columns=result.keys())

local_engine = create_engine("sqlite:///rfam_data.db")
df.to_sql("family_species", con=local_engine, if_exists="replace", index=False)

print(f"Saved {len(df)} rows to rfam_data.db → table 'family_species'")

Saved 14 rows to rfam_data.db → table 'family_species'


In [ ]:
# exercise: execute queries using the syntax discussed above to explore all three databases - 